# Laboratório Multi-Brain CROM-IA (V4.0) 🧠
## Setup P/ NVIDIA A100 - Treinamento Modular Sequencial

Este Notebook fará automaticamente:
1. Montagem do Google Drive.
2. Instalação Unsloth.
3. Loop de Treinamento: Lendo seus datasets V4 (Palavras, Frases, Parágrafos).
4. Exportação nativa GGUF LoRAs (múltiplos micro-cérebros) pro seu Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/CROM-V4.0/MicroCerebros', exist_ok=True)
print("✅ Drive Montado com Sucesso")

In [ ]:
# [OPCIONAL] TÚNEL SSH REVERSO (Assuma o controle pelo Terminal Local)
# Esse bloco fará a A100 comunicar direto com sua tela preta do i5.
!pip install colab_ssh --upgrade -q
from colab_ssh import launch_ssh_cloudflared
launch_ssh_cloudflared(password="crom123")
print("\n✅ TÚNEL ATIVO! Você verá a string de conexão SSH logo acima para colar no seu terminal do Linux.")


In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets

### Orquestrador de Treinos O(N)

In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from datasets import load_dataset
import torch
import gc

max_seq_length = 2048
qwen_base = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"

def treinar_micro_cerebro(nome_cerebro, path_jsonl_dataset):
    print(f"\n🚀 [INICIANDO] Fábrica do Cérebro: {nome_cerebro}")
    
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = qwen_base,
        max_seq_length = max_seq_length,
        dtype = None, load_in_4bit = True,
    )
    
    # Adaptador fresquinho a cada loop
    model = FastLanguageModel.get_peft_model(
        model,
        r = 32,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha = 64,
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
    )
    
    dataset_treino = load_dataset("json", data_files=path_jsonl_dataset, split="train")
    
    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = dataset_treino,
        dataset_text_field = "output",
        max_seq_length = max_seq_length,
        dataset_num_proc = 2,
        args = TrainingArguments(
            per_device_train_batch_size = 4, # A100 permite folga
            gradient_accumulation_steps = 4,
            warmup_steps = 20,
            max_steps = 500, # Treino focado e rapido por Plugin
            learning_rate = 2e-5,
            fp16 = not is_bfloat16_supported(),
            bf16 = is_bfloat16_supported(),
            logging_steps = 50,
            optim = "adamw_8bit",
            lr_scheduler_type = "cosine",
            seed = 3407,
            output_dir = f"/content/drive/MyDrive/CROM-V4.0/MicroCerebros/{nome_cerebro}/tmp",
        ),
    )
    
    trainer.train()
    
    target_dir = f"/content/drive/MyDrive/CROM-V4.0/MicroCerebros/{nome_cerebro}"
    print(f"\n🗜️ Salvando GGUF Adapter 4-bit para {nome_cerebro}...")
    model.save_pretrained_gguf(target_dir, tokenizer, quantization_method = "q4_k_m")
    
    # Descarrega da Memória VRAM da A100 para não estourar no próximo loop
    del model
    del trainer
    gc.collect()
    torch.cuda.empty_cache()
    print(f"✅ LoRA Cérebro {nome_cerebro} criado e memória zerada!")

### A Linha de Produção (Execute Isto Após Fazer o Upload dos seus .jsonl gerados localmente!)

In [ ]:
# Simulação de Array de Cérebros (1 Adaptador LoRA por Área do Conhecimento).
# Cada adaptador conterá sub-regras de Hierarquia (W, F, P) internas e imunizadas.
# Faça o upload dos arquivos dataset_P_Hibrido.jsonl etc gerados!

cerebros_a_treinar = [
    {"nome": "Cerebro_Python", "arquivo": "dataset_P_Hibrido.jsonl"},
    {"nome": "Cerebro_Medicina", "arquivo": "dataset_M_Hibrido.jsonl"},
    {"nome": "Cerebro_Geral_PTBR", "arquivo": "dataset_G_Hibrido.jsonl"},
]

for cer in cerebros_a_treinar:
    if os.path.exists(cer["arquivo"]):
        treinar_micro_cerebro(cer["nome"], cer["arquivo"])
    else:
        print(f"⚠️ Ignorando {cer[nome]} pois não encontrei o arquivo {cer[arquivo]}")

print("\n\n🎉 TODA A FÁBRICA FECHOU SESSÃO! VÁ PARA SEU COMPUTADOR BAIXAR OS .GGUF!")